In [0]:
# Databricks notebook source
# ============================================================
# WORKFLOW 1 — BRONZE LAYER
# Runs all bronze jobs in parallel.
# Silver jobs are triggered INSIDE the ETL raw_to_bronze notebook
# via trigger_jobs field in each bronze job JSON config.
# ============================================================


In [0]:
# MAGIC %md
# MAGIC ## ⚙️ Workflow 1 — Bronze
# MAGIC 
# MAGIC - All bronze jobs (`10001`–`10010`) run 
# MAGIC - Each bronze job's `ETL_NB` path comes from the job JSON
# MAGIC - Silver jobs are triggered **internally** by the ETL notebook using `trigger_jobs`
# MAGIC - No hardcoded notebook paths or job IDs in this orchestrator


In [0]:
# COMMAND ----------
import json, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.functions import date_format, current_timestamp

dbutils.widgets.text("partition", "")
partition = dbutils.widgets.get("partition")
if not partition:
    partition = str(spark.range(1).select(
        date_format(current_timestamp(), "yyyyMMddHHmmssSSS")
        .cast("bigint").alias("p")
    ).collect()[0]["p"])

BASE_PATH  = "/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform"
JOBS_PATH  = f"{BASE_PATH}/jobs/bronze"

print(f"{'='*60}")
print(f"  WORKFLOW 1 — BRONZE")
print(f"  Partition  : {partition}")
print(f"  Jobs path  : {JOBS_PATH}")
print(f"{'='*60}")


In [0]:
# COMMAND ----------
# MAGIC %md ## Discover & Load Bronze Job Configs
# MAGIC
# MAGIC Reads all JSON files from `jobs/bronze/` — no hardcoded job IDs.
# MAGIC Each JSON contains `ETL_NB` and `trigger_jobs` fields.


In [0]:
# COMMAND ----------
# ── Discover all bronze job configs dynamically ───────────────────────
bronze_jobs = []

for fname in sorted(os.listdir(JOBS_PATH)):
    if not fname.endswith(".json"):
        continue
    with open(f"{JOBS_PATH}/{fname}") as f:
        job = json.load(f)
    job["partition"] = partition
    bronze_jobs.append(job)

print(f"  Discovered {len(bronze_jobs)} bronze job(s):\n")
print(f"  {'job_id':<8} {'job_name':<45} {'ETL_NB':<35} trigger_jobs")
print("  " + "─"*110)

for j in bronze_jobs:
    triggers = [str(t['job_id']) for t in j.get('trigger_jobs', [])]
    print(f"  {j['job_id']:<8} {j['job_name']:<45} {j['ETL_NB']:<35} {', '.join(triggers)}")


In [0]:
# COMMAND ----------
# MAGIC %md ## Execute All Bronze Jobs in Parallel
# MAGIC
# MAGIC Each job uses its own `ETL_NB` from the JSON config.
# MAGIC Silver triggering is handled inside the ETL notebook.


In [0]:
# COMMAND ----------
def run_bronze_job(job_config):
    job_id   = job_config["job_id"]
    job_name = job_config["job_name"]
    # ── ETL_NB path comes from job config — not hardcoded ──────────────
    etl_nb   = f"{BASE_PATH}/{job_config['ETL_NB']}"
    triggers = [t['job_id'] for t in job_config.get('trigger_jobs', [])]
    try:
        print(f"  [{job_id}] 🔶 Starting Bronze — {job_name}")
        print(f"  [{job_id}]    ETL NB   : {job_config['ETL_NB']}")
        if triggers:
            print(f"  [{job_id}]    Will trigger silver: {triggers}")
        try:
            dbutils.notebook.run(
                etl_nb, 500,
                {"job_parameters": json.dumps(job_config)}
            )
        except Exception as e:
            dbutils.notebook.run(
                etl_nb, 500,
                {"job_parameters": json.dumps(job_config)}
            )
        print(f"  [{job_id}] ✅ Bronze complete — {job_name}")
        return job_id, job_name, "SUCCESS"
    except Exception as e:
        err = f"FAILED: {str(e)}"
        print(f"  [{job_id}] ❌ {job_name}: {err}")
        return job_id, job_name, err


In [0]:
# COMMAND ----------
print(f"\n  Running {len(bronze_jobs)} bronze jobs in parallel...\n")

results = []

for job in bronze_jobs:
    print(f"  [{job['job_id']}] {job['job_name']}")
    print(f"  [{job['job_id']}]    ETL NB   : {job['ETL_NB']}")
    output1, output2, output3 = run_bronze_job(job)
    results.append((output1, output2, output3))


In [0]:
# COMMAND ----------
failed    = [(j, n, s) for j, n, s in results if "FAILED" in s]
succeeded = [(j, n, s) for j, n, s in results if s == "SUCCESS"]

print(f"\n{'='*60}")
print(f"  WORKFLOW 1 SUMMARY")
print(f"{'='*60}")
print(f"  Total  : {len(results)}")
print(f"  ✅     : {len(succeeded)}")
print(f"  ❌     : {len(failed)}")
print()
for jid, name, status in sorted(results, key=lambda x: x[0]):
    icon = "✅" if status == "SUCCESS" else "❌"
    print(f"  {icon} [{jid}] {name:<45} {status}")

if failed:
    raise Exception(
        f"Workflow 1 failed — {len(failed)} bronze job(s) failed: "
        f"{[(j, n) for j, n, _ in failed]}"
    )

print(f"\n  ✅ Workflow 1 complete — Bronze done, Silver triggered internally")
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "partition": partition}))
